In [4]:
# 1. Cài đặt các thư viện cần thiết trên Colab
!pip install torch scikit-learn onnx onnxscript

import os
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import datetime

# 2. Chuẩn bị dữ liệu giả lập
X, y = make_classification(n_samples=5000, n_features=4, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

# 3. Định nghĩa mô hình
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.linear = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.linear(x)

lr = 0.005
epochs = 100

model = SimpleNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# 4. Train model
print("--- Đang train model trên GPU/CPU Colab ---")
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

# Đánh giá Accuracy
model.eval()
with torch.no_grad():
    preds = model(X_test_t)
    preds_cls = (preds >= 0.5).float()
    accuracy = (preds_cls.squeeze() == torch.tensor(y_test, dtype=torch.float32)).float().mean().item()

print(f"Train hoàn tất! Accuracy: {accuracy:.4f}")

# 5. Xuất ra file ONNX kèm theo tên version / thời gian cho dễ quản lý
version_name = datetime.datetime.now().strftime("model_v_%Y%m%d_%H%M%S.onnx")
dummy_input = torch.randn(1, 4, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    version_name,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"Đã xuất thành công file: {version_name}")

--- Đang train model trên GPU/CPU Colab ---
Train hoàn tất! Accuracy: 0.8900


/tmp/ipykernel_994/3286741521.py:64: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SimpleNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Đã xuất thành công file: model_v_20261102_020824.onnx


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [6]:
# 1. Định nghĩa mô hình cải tiến (Deep & Robust Network)
class ImprovedNet(nn.Module):
    def __init__(self):
        super(ImprovedNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# 2. Khởi tạo mô hình, loss function, optimizer và scheduler
improved_model = ImprovedNet()
criterion = nn.BCELoss()
optimizer = optim.AdamW(improved_model.parameters(), lr=0.01, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

# 3. Huấn luyện mô hình cải tiến
epochs = 150
print("--- Đang train mô hình cải tiến ---")
improved_model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = improved_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    # Cập nhật learning rate dựa trên loss
    scheduler.step(loss.item())

# 4. Đánh giá Accuracy trên tập test
improved_model.eval()
with torch.no_grad():
    preds = improved_model(X_test_t)
    preds_cls = (preds >= 0.5).float()
    improved_accuracy = (preds_cls.squeeze() == torch.tensor(y_test, dtype=torch.float32)).float().mean().item()

print(f"Huấn luyện hoàn tất! Accuracy mới: {improved_accuracy:.4f} (Trước đó: {accuracy:.4f})")

# 5. Xuất mô hình mới ra file ONNX sử dụng dynamic_shapes để tránh cảnh báo cũ
import onnxscript
version_name_improved = datetime.datetime.now().strftime("model_improved_v_%Y%m%d_%H%M%S.onnx")
dummy_input = torch.randn(1, 4, dtype=torch.float32)

torch.onnx.export(
    improved_model,
    dummy_input,
    version_name_improved,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"Đã xuất thành công file mô hình cải tiến: {version_name_improved}")

--- Đang train mô hình cải tiến ---
Huấn luyện hoàn tất! Accuracy mới: 0.9180 (Trước đó: 0.8900)


/tmp/ipykernel_994/2971378965.py:60: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ImprovedNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ImprovedNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Đã xuất thành công file mô hình cải tiến: model_improved_v_20261102_020932.onnx


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


# cải thiên code
